# Create HFSP Awards from Official Awardee Listing

Creates OpenAlex award rows from the Human Frontier Science Program (HFSP) official public awardee listing.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/hfsp_to_s3.py` to fetch the official paginated HTML listing, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://www.hfsp.org/awardees/awards`  
**S3 location:** `s3a://openalex-ingest/awards/hfsp/hfsp_awards.parquet`  
**Local full-source validation on 2026-05-26:** 4,979 award rows from pages `?page=0` through `?page=248`; page 249 is empty. Coverage: 100% title/year/landing-url, 4,978 program labels (100.0% rounded; one 2000 source row lacks a program label), 4,925 lead awardees/institutions (98.9%), 2,740 abstracts (55.0%), 815 co-awardees after excluding host supervisors, 3,866 rows with host-supervisor metadata preserved in raw fields. Year range 1990-2026. Funding type distribution: 3,866 fellowship rows, 1,113 grant rows. Amount/currency are NULL because HFSP does not publish per-award amounts in the official listing.

**OpenAlex funder mapping:**
- Chosen funder_id: 4320320338
- DOI: `10.13039/501100000854`
- ROR: `https://ror.org/02ebx7v45`
- Duplicate note: OpenAlex also has `F4320307846` / DOI `10.13039/100004412` for HFSP with the same ROR. Crossref's `501100000854` row includes alternate names `International Human Frontier Science Program Organization` and `HFSPO`, so this ingest uses F4320320338 and preserves the duplicate note for audit.

**Mapping summary:** One OpenAlex award row per official HFSP award card. For team grants, the first listed awardee becomes `lead_investigator`, the second listed awardee becomes `co_lead_investigator`, and all listed awardees are placed in `investigators`. Host supervisors are preserved in raw `host_supervisors_raw` / `people_json` but are not treated as co-leads.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hfsp_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hfsp/hfsp_awards.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-26 found 4,979 rows.
SELECT COUNT(*) as total_hfsp_awards
FROM openalex.awards.hfsp_awards_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.hfsp_awards_raw;

In [ ]:
%sql
-- Sample raw HFSP data.
SELECT
    funder_award_id,
    display_name,
    source_year,
    program,
    funding_type,
    lead_person_name,
    lead_institution,
    co_person_name,
    awardee_count,
    host_supervisor_count,
    landing_page_url
FROM openalex.awards.hfsp_awards_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. HFSP's official listing has no per-award amount fields.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'hfsp_awards_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency';

In [ ]:
%sql
-- Confirm amount/date/source coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS rows_with_title,
    COUNT(source_year) AS rows_with_year,
    COUNT(program) AS rows_with_program,
    COUNT(lead_person_name) AS rows_with_lead,
    COUNT(lead_institution) AS rows_with_lead_institution,
    COUNT(description) AS rows_with_description,
    COUNT(amount) AS rows_with_amount,
    MIN(TRY_CAST(source_year AS INT)) AS min_year,
    MAX(TRY_CAST(source_year AS INT)) AS max_year
FROM openalex.awards.hfsp_awards_raw;

In [ ]:
%sql
-- Native-key inspection.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT source_hash) AS distinct_source_hashes,
    COUNT(DISTINCT CONCAT(source_page_number, ':', source_card_index)) AS distinct_page_card_positions
FROM openalex.awards.hfsp_awards_raw;

In [ ]:
%sql
-- Program and funding-type distribution.
SELECT funding_type, program, COUNT(*) AS cnt
FROM openalex.awards.hfsp_awards_raw
GROUP BY funding_type, program
ORDER BY cnt DESC;

In [ ]:
%sql
-- Awardee and host-supervisor coverage.
SELECT
    COUNT(*) AS total,
    SUM(TRY_CAST(awardee_count AS INT)) AS total_awardee_mentions,
    SUM(TRY_CAST(host_supervisor_count AS INT)) AS total_host_supervisor_mentions,
    COUNT(CASE WHEN TRY_CAST(host_supervisor_count AS INT) > 0 THEN 1 END) AS rows_with_host_supervisors,
    COUNT(co_person_name) AS rows_with_co_awardees,
    COUNT(investigators_json) AS rows_with_investigators_json
FROM openalex.awards.hfsp_awards_raw;

In [ ]:
%sql
-- Verify the JSON parser shape for the all-awardee array.
WITH parsed AS (
    SELECT
        funder_award_id,
        from_json(
            investigators_json,
            'ARRAY<STRUCT<person_name:STRING,given_name:STRING,family_name:STRING,nationality:STRING,role:STRING,institution:STRING,city:STRING,country:STRING>>'
        ) AS awardees
    FROM openalex.awards.hfsp_awards_raw
    WHERE investigators_json IS NOT NULL
    LIMIT 5
)
SELECT
    funder_award_id,
    size(awardees) AS awardee_count,
    try_element_at(awardees, 1).person_name AS first_awardee,
    try_element_at(awardees, 1).institution AS first_institution
FROM parsed;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Fail fast if the chosen HFSP funder is missing from openalex.common.funder.
-- Duplicate note: F4320307846 also exists in public OpenAlex with the same ROR; this ingest intentionally uses F4320320338.
SELECT assert_true(
    COUNT(*) = 1,
    'Expected exactly one openalex.common.funder row for HFSP F4320320338'
) AS funder_row_exists
FROM openalex.common.funder
WHERE funder_id = 4320320338;

## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hfsp_awards
USING delta
AS
WITH
hfsp_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320320338
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS parsed_end_date,
        TRY_CAST(source_year AS INT) AS parsed_year,
        from_json(
            investigators_json,
            'ARRAY<STRUCT<person_name:STRING,given_name:STRING,family_name:STRING,nationality:STRING,role:STRING,institution:STRING,city:STRING,country:STRING>>'
        ) AS awardees
    FROM openalex.awards.hfsp_awards_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        COALESCE(NULLIF(TRIM(g.funding_type), ''), 'grant') as funding_type,

        COALESCE(NULLIF(TRIM(g.program), ''), 'Unlabeled HFSP row') as funder_scheme,

        'hfsp_awards_listing' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_year) as start_year,
        COALESCE(YEAR(g.parsed_end_date), g.parsed_year) as end_year,

        CASE
            WHEN g.lead_person_name IS NULL OR TRIM(g.lead_person_name) = '' THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                NULLIF(TRIM(g.lead_given_name), '') as given_name,
                NULLIF(TRIM(g.lead_family_name), '') as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    NULLIF(TRIM(g.lead_institution), '') as name,
                    NULLIF(TRIM(g.lead_country), '') as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as lead_investigator,

        CASE
            WHEN g.co_person_name IS NULL OR TRIM(g.co_person_name) = '' THEN
                CAST(NULL AS STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >)
            ELSE struct(
                NULLIF(TRIM(g.co_given_name), '') as given_name,
                NULLIF(TRIM(g.co_family_name), '') as family_name,
                CAST(NULL AS STRING) as orcid,
                g.parsed_start_date as role_start,
                struct(
                    NULLIF(TRIM(g.co_institution), '') as name,
                    NULLIF(TRIM(g.co_country), '') as country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                ) as affiliation
            )
        END as co_lead_investigator,

        CASE
            WHEN g.awardees IS NULL OR size(g.awardees) = 0 THEN
                CAST(NULL AS ARRAY<STRUCT<
                    given_name:STRING,
                    family_name:STRING,
                    orcid:STRING,
                    role_start:DATE,
                    affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
                >>)
            ELSE TRANSFORM(
                g.awardees,
                x -> struct(
                    NULLIF(TRIM(x.given_name), '') as given_name,
                    NULLIF(TRIM(x.family_name), '') as family_name,
                    CAST(NULL AS STRING) as orcid,
                    g.parsed_start_date as role_start,
                    struct(
                        NULLIF(TRIM(x.institution), '') as name,
                        NULLIF(TRIM(x.country), '') as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            )
        END as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN hfsp_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 124

In [ ]:
%sql
-- Remove previous HFSP award-listing data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hfsp_awards_listing' AND priority = 124;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    124 as priority
FROM openalex.awards.hfsp_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_hfsp_awards
FROM openalex.awards.hfsp_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.hfsp_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.given_name AS lead_given,
    lead_investigator.family_name AS lead_family,
    lead_investigator.affiliation.name AS lead_affiliation,
    co_lead_investigator.family_name AS co_family,
    size(investigators) AS investigator_count,
    landing_page_url
FROM openalex.awards.hfsp_awards
LIMIT 10;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.hfsp_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only F4320320338.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.hfsp_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator) as has_lead,
    COUNT(co_lead_investigator) as has_co_lead,
    COUNT(investigators) as has_investigators_array,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) as pct_title,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(lead_investigator), COUNT(*)) * 100.0, 1) as pct_lead,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_amount
FROM openalex.awards.hfsp_awards;

In [ ]:
%sql
-- Amount/currency fail-fast. Expected: 0% amount coverage by source authority.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies
FROM openalex.awards.hfsp_awards;

In [ ]:
%sql
-- Year distribution.
SELECT start_year, COUNT(*) as cnt
FROM openalex.awards.hfsp_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;

In [ ]:
%sql
-- Funding type and scheme distribution.
SELECT funding_type, funder_scheme, COUNT(*) as cnt
FROM openalex.awards.hfsp_awards
GROUP BY funding_type, funder_scheme
ORDER BY cnt DESC;

In [ ]:
%sql
-- Lead-affiliation country distribution from the source strings.
SELECT lead_investigator.affiliation.country AS country, COUNT(*) AS cnt
FROM openalex.awards.hfsp_awards
WHERE lead_investigator.affiliation.country IS NOT NULL
GROUP BY lead_investigator.affiliation.country
ORDER BY cnt DESC
LIMIT 30;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 124.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hfsp_awards_listing' AND priority = 124
GROUP BY priority, provenance;